## **DO NOT rename or change the signature of these functions. Your code must be in the 3rd cell of the notebook, otherwise the tests will fall.**

# Homework: AI Agents

## Instructions
1. **"Template" cell** — run it first, do not modify.
2. **"Tasks" cell** — write your code where you see `# YOUR CODE HERE`.
3. Run the open examples and make sure all say `OK`.
4. Submit the notebook with saved outputs.

In [ ]:
import os
os.environ["YANDEX_CLOUD_API_KEY"] = ""
os.environ["YANDEX_CLOUD_FOLDER"] = ""

In [70]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

YANDEX_CLOUD_API_KEY = os.environ.get("YANDEX_CLOUD_API_KEY")
YANDEX_CLOUD_FOLDER = os.environ.get("YANDEX_CLOUD_FOLDER")
YANDEX_CLOUD_MODEL = "gpt-oss-20b/latest"

llm = ChatOpenAI(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
    default_headers={"OpenAI-Project": YANDEX_CLOUD_FOLDER},
    temperature=0,
)

In [ ]:
# from langchain_ollama import ChatOllama

# os.environ["OLLAMA_API_KEY"] = ""

# llm = ChatOllama(
#     model="gpt-oss:20b",
#     temperature=0,
#     base_url="https://ollama.com",
#     client_kwargs={
#         "headers": {
#             "Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}",
#         }
#     },
# )

In [71]:
# Connection check
resp = llm.invoke([
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Reply with one word: working?"),
])

print(f'Test: {resp.content}')

Test: working


In [72]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:100]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
# print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


Template loaded.
  Catalog: 10 products
  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool
  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage


## Task 1. Tool-Calling Agent (ReAct loop)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1.1. Define SHOP_TOOLS_SCHEMA — tool descriptions for the LLM.
#
# Below are stub functions with signatures but no descriptions.
# The LLM needs to understand what each tool does and what its parameters mean.
#
# Task: add a docstring (description + Args) to each function.
# The convert_to_openai_tool() function from the template will generate the JSON schema automatically.
# For docstring format details, see Google-style docstrings.

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """Searches products in the catalog using optional filters.

    Args:
        query: Free-text query such as product type, features, or brand words.
        category: Product category to filter by, for example headphones or mouse.
        brand: Brand name to filter by.
        max_price: Maximum allowed price.
        sort_by: Sorting mode. Supported values include "price_asc" and "rating_desc".

    Returns:
        A list of matching products.
    """

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """Adds a product to the shopping cart.

    Args:
        product_id: Catalog product id to add.
        quantity: Number of units to add.

    Returns:
        A dict describing whether the operation succeeded.
    """

# YOUR CODE HERE: generate the schema
SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]

REACT_SYSTEM_PROMPT = """You are a shopping assistant for a small electronics store. 
The store sells headphones, earbuds, keyboards, mice, and e-readers

Work in a ReAct-style loop internally. Follow the TAO loop:
THOUGHT: Analyze the request, plan the steps
ACTION: Call the appropriate tool to get information
OBSERVATION: Analyze the result, determine if further actions are needed

IMPORTANT:
- Use tools when you need to search products or add items to the cart
- Use search_products before making specific recommendations or choosing the cheapest/best item
- Use add_to_cart only when the user asks to add something to the cart
- After receiving tool results, continue reasoning and respond concisely
"""

# 1.2. Implement run_shopping_agent — a ReAct shop agent.
def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """

    messages = [
        SystemMessage(content=REACT_SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ]

    for _ in range(5):
        ai_msg = llm_chat(messages, SHOP_TOOLS_SCHEMA)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg.content
            # if isinstance(ai_msg.content, str) else str(ai_msg.content)

        for call in ai_msg.tool_calls:
            name = call["name"]
            args = call.get("args", {}) or {}

            if name == "search_products":
                result = tools.search_products(**args)
                state.last_results = result
            elif name == "add_to_cart":
                result = tools.add_to_cart(state, **args)
            else:
                result = {"ok": False, "error": f"Unknown tool: {name}"}

            tracer.record(name, args, result)

            messages.append(
                ToolMessage(
                    content=json.dumps(result, ensure_ascii=False),
                    tool_call_id=call["id"],
                )
            )

    return "I could not complete the request in the allowed number of steps."

In [59]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


=== Tool Call Trace ===
  1. search_products({"category": "headphones", "max_price": 150, "sort_by": "price_asc"})
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
OK 1.A
OK 1.B
OK 1.C: 'NuPhy Air75' (rating 4.6)


In [43]:
#  "Find a wireless keyboard with the best rating and add it to cart",
_t1c.print_trace()

=== Tool Call Trace ===
  1. search_products({"category": "keyboard", "sort_by": "rating_desc"})
     -> [{"id": "p9", "name": "NuPhy Air75", "category": "keyboard", "brand": "NuPhy", "price": 139, "color"
  2. add_to_cart({"product_id": "p9", "quantity": 1})
     -> {"ok": true, "cart_size": 1}


In [44]:
_s1c

ShopState(cart=[{'product_id': 'p9', 'name': 'NuPhy Air75', 'price': 139, 'quantity': 1}], last_results=[{'id': 'p9', 'name': 'NuPhy Air75', 'category': 'keyboard', 'brand': 'NuPhy', 'price': 139, 'color': 'gray', 'rating': 4.6, 'tags': ['wireless', 'mechanical', 'low-profile']}, {'id': 'p8', 'name': 'Keychron K2', 'category': 'keyboard', 'brand': 'Keychron', 'price': 89, 'color': 'black', 'rating': 4.5, 'tags': ['wireless', 'mechanical', 'compact']}])

In [45]:
_r1c

'✅ The **NuPhy Air75** wireless keyboard (rating\u202f4.6) has been added to your cart. Let me know if you’d like to check out or add anything else!'

## TASK 2. Memory Agent

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")
# Recommended profile fields:
#   name       — user name
#   brand      — preferred brand
#   max_price  — maximum price
#   color      — preferred color
#   category   — category of interest

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    if path.exists():
        try:
            return json.loads(path.read_text())
        except json.JSONDecodeError:
            # Corrupted file — reset to empty
            save_profile({})
            return {}
    return {}

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    path.write_text(json.dumps(profile, indent=2, ensure_ascii=False))

# Initialize
# save_profile({})

def update_profile(key: str, value: str, path: Path = PROFILE_PATH) -> str:
    """Update a field in the users persistent profile.

    Recommended field names: name, brand, max_price, color, category.

    Args:
        key: Field name to update (e.g. 'brand')
        value: New value for the field (e.g. 'Apple')

    Returns:
        Confirmation message.
    """
    print(f"[TOOL] update_profile(key='{key}', value='{value}')")
    profile = load_profile(path)
    profile[key] = value
    save_profile(profile, path)
    return f"Profile updated: {key} = '{value}'"

# SHOP_TOOLS_SCHEMA_WITH_MEMORY = SHOP_TOOLS_SCHEMA.append(convert_to_openai_tool(update_profile))

SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
    convert_to_openai_tool(update_profile)
]


MEMORY_SYSTEM_PROMPT = REACT_SYSTEM_PROMPT + """
You also have access to the user's persistent profile and the conversation history.

Memory rules:
- Save any user preference or personal detail to the persistent profile.
- If the user shares their name, preferred brand, budget, preferred color, product category, or any other stable shopping preference, immediately call update_profile.
- Call update_profile once per field.
- Do not ask for confirmation before saving.
- Recommended profile fields: name, brand, max_price, color, category.
- At the start of each conversation, use the user's current profile to personalize your response.
- If the user asks about information they shared earlier, use the profile when available.
- Use conversation history to handle follow-up requests like "add the first one found".
- Do not invent profile values that are not present in the profile or conversation history.
"""


def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple[str, list]:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    Hint: save ALL messages to history (HumanMessage, AIMessage, ToolMessage),
    so the agent knows what was found in the next turn.
    """

    profile = load_profile(profile_path)
    if profile:
        profile_text = "\n".join(f"  {k}: {v}" for k, v in profile.items())
        full_prompt = MEMORY_SYSTEM_PROMPT + f"\n## Current User Profile\n{profile_text}\n"
    else:
        full_prompt = MEMORY_SYSTEM_PROMPT + "\n## Current User Profile\n(empty — no data saved yet)\n"

    user_msg = HumanMessage(content=user_message)

    updated_history = history.copy()
    updated_history.append(user_msg)

    messages = [
        SystemMessage(content=full_prompt),
        *updated_history,
    ]

    for _ in range(5):
        ai_msg = llm_chat(messages, SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        messages.append(ai_msg)
        updated_history.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg.content, updated_history

        for call in ai_msg.tool_calls:
            name = call["name"]
            args = call.get("args", {}) or {}

            if name == "search_products":
                result = tools.search_products(**args)
                state.last_results = result
            elif name == "add_to_cart":
                result = tools.add_to_cart(state, **args)
            elif name == "update_profile":
                result = update_profile(key=args["key"], value=args["value"], path=profile_path)

            else:
                result = {"ok": False, "error": f"Unknown tool: {name}"}

            tracer.record(name, args, result)

            tool_msg = ToolMessage(
                    content=json.dumps(result, ensure_ascii=False),
                    tool_call_id=call["id"],
                    )
            messages.append(tool_msg)
            updated_history.append(tool_msg)

    return "I could not complete the request in the allowed number of steps.", updated_history

In [55]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")

[TOOL] update_profile(key='name', value='Anna')
[TOOL] update_profile(key='brand', value='Sony')
[TOOL] update_profile(key='max_price', value='200')
OK 2.A
OK 2.B
OK 2.C: added 'Sony WH-CH720N'


In [56]:
_t2a.print_trace()

=== Tool Call Trace ===
  1. update_profile({"key": "name", "value": "Anna"})
     -> "Profile updated: name = 'Anna'"
  2. update_profile({"key": "brand", "value": "Sony"})
     -> "Profile updated: brand = 'Sony'"
  3. update_profile({"key": "max_price", "value": "200"})
     -> "Profile updated: max_price = '200'"


In [58]:
_h2c

[HumanMessage(content='Find wireless headphones under 150 dollars', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 676, 'total_tokens': 748, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt://gpt-oss-20b/latest', 'system_fingerprint': None, 'id': 'chatcmpl-9fd8a923-8d8a-4de2-aa07-2ea3f9ea3b46', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da405-62da-7dc2-8329-88cbe705071f-0', tool_calls=[{'name': 'search_products', 'args': {'query': 'wireless headphones', 'category': 'headphones', 'max_price': 150, 'sort_by': 'price_asc'}, 'id': 'chatcmpl-tool-5923f8b6ee574267b09cc088ac46132e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 676, 'output_tokens': 72, 'total_tokens': 748, 'input_token_details': {'cache_r

In [57]:
_t2c2.print_trace()

=== Tool Call Trace ===
  1. add_to_cart({"product_id": "p2", "quantity": 1})
     -> {"ok": true, "cart_size": 1}


## TASK 3. Multi-Agent System

### Prompts

In [65]:
RETRIEVER_SYSTEM_PROMPT = """You are retriever_agent in a shopping multi-agent system.

Your only job is to prepare and execute ONE search_products tool call for the user's request.

Rules:
- Call search_products exactly once.
- Do not answer with prose unless the tool cannot be called.
- Do not analyze pros or cons.
- Do not rank products.
- Do not add anything to cart.

When building tool arguments:
- Extract structured constraints from the user's request when possible:
  - category
  - brand
  - max_price
  - sort_by
- Put into query only the product-related keywords that help retrieval, such as:
  - wireless
  - mechanical
  - compact
  - noise-cancelling
- Do NOT put generic instruction words into query, such as:
  - find
  - show
  - best
  - recommend
  - add to cart
  - dollars
  - under
- If the user mentions a budget, pass max_price as a number.
- If the user asks for the best option, prefer sort_by="rating_desc".
- If the user asks for the cheapest or budget option, prefer sort_by="price_asc".
- If category or brand is not clearly specified, leave them empty rather than guessing.
- Search for products relevant to the request and return up to 5 candidates after tool execution.
"""

In [66]:
PROS_SYSTEM_PROMPT = """You are pros_agent in a shopping multi-agent system.

Your task:
- For each candidate product, write 1-2 short sentences describing its advantages.
- Use only the provided product data and the user query.
- Do not invent features that are not present in the data.

Output rules:
- Return ONLY valid JSON.
- No markdown.
- No code fences.
- Format: {"product_id": "pros text", "product_id_2": "pros text"}

Focus on advantages such as:
- strong rating
- good price for the request
- useful tags/features
- relevant brand/category fit
"""

In [67]:
CONS_SYSTEM_PROMPT = """You are cons_agent in a shopping multi-agent system.

Your task:
- For each candidate product, write 1-2 short sentences describing its disadvantages.
- Use only the provided product data and the user query.
- Do not invent drawbacks that are not supported by the data.

Output rules:
- Return ONLY valid JSON.
- No markdown.
- No code fences.
- Format: {"product_id": "cons text", "product_id_2": "cons text"}

Focus on disadvantages such as:
- higher price compared with other candidates
- lower rating compared with other candidates
- missing useful tags/features implied by the user request
- possible mismatch with the requested category, brand, budget, or use case

Be honest and specific, but stay grounded in the given data only.
"""

In [68]:
COORDINATOR_SYSTEM_PROMPT = """You are the coordinator of a shopping multi-agent system.

You have 4 specialists:
- retriever_agent: searches for products
- pros_agent: analyzes product advantages
- cons_agent: analyzes product disadvantages
- ranker_agent: selects the best product using deterministic logic

Your job is to orchestrate the workflow and decide whether the final selected product should be added to cart.

Execution policy:
1. The retriever_agent searches for up to 5 relevant products.
2. The pros_agent writes short pros for each candidate.
3. The cons_agent writes short cons for each candidate.
4. The ranker_agent chooses the best product.
5. Only after the best product is selected, decide whether to add it to cart.

Cart decision rules:
- Use the add_to_cart tool only if the user clearly asks to add, buy, or purchase the product.
- Do not call add_to_cart if the user only asks to search, compare, recommend, or rank products.
- If the user requests adding to cart, call add_to_cart exactly once.
- The Python controller will provide the selected product_id separately, so you only need to decide whether a cart action is needed and what quantity to use.
- If the quantity is not specified, use quantity=1.
- Return a tool call when cart action is needed. Otherwise return no tool call.

Do not analyze products yourself.
Do not invent product IDs.
Do not call any tool unless the user intent is clearly to add the selected item to cart.
"""

### Implementation

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Implement a system of four agents + an orchestrator.
# Goal — find the best product and honestly describe its pros and cons.
# Agents work in a chain via a shared AgentContext object (defined in the template).
#
# RetrieverAgent (LLM + tools)
#   Searches for up to 5 relevant products via search_products.
#   Fills ctx.candidates and ctx.max_price.
#   Important: only pass the search tool (not add_to_cart).
#
# ProsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of pros.
#   Fills ctx.pros (dict: product_id -> pros string).
#   Records an "analyze_pros" call in tracer.
#
# ConsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of cons.
#   Fills ctx.cons (dict: product_id -> cons string).
#   Records an "analyze_cons" call in tracer.
#
# RankerAgent (no LLM — logic only)
#   Picks the best product from ctx.candidates:
#     - Filters by ctx.max_price (if set)
#     - Among remaining: highest rating; if tied — lowest price
#   Records a "rank_candidates" call in tracer. Fills ctx.best.
#
# CoordinatorAgent (orchestrator)
#   Runs agents in a chain, maintains a trace list.
#   Trace keys: "delegate_retriever", "delegate_pros", "delegate_cons",
#               "delegate_ranker", "delegate_cart".
#   No CartAgent needed — if the user asks to add to cart,
#   CoordinatorAgent does it itself via tools.add_to_cart after ranking.
#   Returns AgentResult with response, trace, and context.
#   The response should include: product name, price, rating, pros and cons.


SHOP_TOOLS_SCHEMA_RETRIEVER = [
    convert_to_openai_tool(search_products),
]
SHOP_TOOLS_SCHEMA_COORD = [
    convert_to_openai_tool(add_to_cart),
]

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""

        def extract_max_price_from_query(query: str) -> float | None:
            import re
            q = query.lower().replace(",", " ")

            patterns = [
                r"(?:under|below|up to|less than|at most|max(?:imum)?)\s*\$?\s*(\d+(?:\.\d+)?)",
                r"\$?\s*(\d+(?:\.\d+)?)\s*(?:dollars?|usd)\s*(?:or less|or below|or cheaper)?",
            ]

            for pattern in patterns:
                match = re.search(pattern, q)
                if match:
                    try:
                        return float(match.group(1))
                    except (TypeError, ValueError):
                        pass

            return None
        
        fallback_price = extract_max_price_from_query(ctx.query)

        # Отправил ctx.query в LLM с единственным tool search_products
        messages = [SystemMessage(content=RETRIEVER_SYSTEM_PROMPT),
                    HumanMessage(content=f"User query: {ctx.query}\n")]

        # LLM должна собрать (распарсить) параметры поиска
        ai_msg = llm_chat(messages, SHOP_TOOLS_SCHEMA_RETRIEVER)

        if not ai_msg.tool_calls:
            ctx.max_price = fallback_price
            return ctx

        # Взял args и выполнил tools.search_products(**args)
        for call in ai_msg.tool_calls:
            name = call["name"]
            args = call.get("args", {}) or {}

            if name == "search_products":
                # Если модель вызовет search_products() без max_price, то budget потеряется. 
                # То нужен fallback: ctx.max_price - можно дополнительно извлекать из ctx.query, если в args его нет.
                # Так как потом RankerAgent по условию должен принимать ctx.max_price
                if args.get("max_price") is None and fallback_price is not None:
                    args["max_price"] = fallback_price

                result = tools.search_products(**args)
                state.last_results = result
                ctx.candidates = result[:5]
                ctx.max_price = args.get("max_price")
            else:
                result = {"ok": False, "error": f"Unknown tool: {name}"}

            tracer.record(name, args, result)
            break

        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""

        if not ctx.candidates:
            ctx.pros = {}
            tracer.record("analyze_pros", {"candidate_ids": []}, ctx.pros)
            return ctx

        #  candidates это список dict, его надо отдельно сериализовать
        candidates_json = json.dumps(ctx.candidates, ensure_ascii=False, indent=2)

        # Отправил ctx.query и список кандидатов
        messages = [SystemMessage(content=PROS_SYSTEM_PROMPT),
                    HumanMessage(
                    f"User query: {ctx.query}\n\n"
                    f"Candidates JSON:\n{candidates_json}"
                                 )]

        # LLM должна для каждого продукта в ctx.candidates написать 1-2 предложения с описанием преимуществ.
        ai_msg = llm_chat(messages)

        content = ai_msg.content if isinstance(ai_msg.content, str) else str(ai_msg.content)
        content = content.strip()
        if content.startswith("```"):
            content = content.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        
        # После llm_chat(...) сделать json.loads(ai_msg.content) и положить в ctx.pros.
        ctx.pros = json.loads(content)
        args={"candidate_ids": [item["id"] for item in ctx.candidates]}
        tracer.record("analyze_pros", args, ctx.pros)
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""

        if not ctx.candidates:
            ctx.cons = {}
            tracer.record("analyze_pros", {"candidate_ids": []}, ctx.cons)
            return ctx

        #  candidates это список dict, его надо отдельно сериализовать
        candidates_json = json.dumps(ctx.candidates, ensure_ascii=False, indent=2)

        # Отправил ctx.query и список кандидатов
        messages = [SystemMessage(content=CONS_SYSTEM_PROMPT),
                    HumanMessage(
                    f"User query: {ctx.query}\n\n"
                    f"Candidates JSON:\n{candidates_json}"
                                 )]

        # LLM должна для каждого продукта в ctx.candidates написать 1-2 предложения с описанием преимуществ.
        ai_msg = llm_chat(messages)

        content = ai_msg.content if isinstance(ai_msg.content, str) else str(ai_msg.content)
        content = content.strip()
        if content.startswith("```"):
            content = content.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

        # После llm_chat(...) сделать json.loads(ai_msg.content) и положить в ctx.cons.
        ctx.cons = json.loads(content)
        args={"candidate_ids": [item["id"] for item in ctx.candidates]}
        tracer.record("analyze_cons", args, ctx.cons)
        return ctx

class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        
        candidates = ctx.candidates

        # Фильтрует по ctx.max_price (если задано)
        if ctx.max_price is not None:
            candidates = [
                item for item in candidates
                if item.get("price") is not None and float(item["price"]) <= float(ctx.max_price)
            ]

        # Если не нашлось
        if not candidates:
            ctx.best = None
            tracer.record(
                "rank_candidates",
                {"max_price": ctx.max_price, "candidate_ids": [item["id"] for item in ctx.candidates]},
                None,
            )
            return ctx

        # Если не нашлось
        # Среди оставшихся: наивысший рейтинг; если одинаковый — самая низкая цена
        best_item = candidates[0]

        for item in candidates[1:]:
            item_rating = float(item["rating"])
            best_rating = float(best_item["rating"])
            item_price = float(item["price"])
            best_price = float(best_item["price"])

            if item_rating > best_rating:
                best_item = item
            elif item_rating == best_rating and item_price < best_price:
                best_item = item

        ctx.best = best_item

        tracer.record(
            "rank_candidates",
            {"max_price": ctx.max_price, "candidate_ids": [item["id"] for item in candidates]},
            ctx.best,
        )

        return ctx


class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    # def run(self, user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentResult:
    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        
        tracer = ToolTracer()
        
        # create agents context
        ctx = AgentContext(query=user_message)

        # ведёт отдельный trace со строками delegate_*
        trace = []

        # call subagents
        trace.append("delegate_retriever")
        ctx = self.retriever.run(ctx, state, tools, tracer)

        trace.append("delegate_pros")
        ctx = self.pros_agent.run(ctx, tracer)

        trace.append("delegate_cons")
        ctx = self.cons_agent.run(ctx, tracer)

        trace.append("delegate_ranker")
        ctx = self.ranker.run(ctx, tracer)

        # Если нашли лучший товар
        if ctx.best is not None:
            # Отправил ctx.query в LLM с единственным tool add_to_cart
            messages = [SystemMessage(content=COORDINATOR_SYSTEM_PROMPT),
                        HumanMessage(content=f"User query: {ctx.query}\n")]

            # LLM должна распарсить есть ли намерение купить товар и количество
            ai_msg = llm_chat(messages, SHOP_TOOLS_SCHEMA_COORD)

            if ai_msg.tool_calls:
                # Взял args и выполнил tools.add_to_cart(**args)
                for call in ai_msg.tool_calls:
                    name = call["name"]
                    args = call.get("args", {}) or {}

                    quantity = args.get("quantity", 1)
                    product_id = ctx.best["id"]

                    # если quantity не число или меньше 1, использовать 1
                    try:
                        quantity = int(quantity)
                    except (TypeError, ValueError):
                        quantity = 1

                    if quantity < 1:
                        quantity = 1

                    if name == "add_to_cart":
                        result = tools.add_to_cart(state, product_id=product_id, quantity=quantity)
                        trace.append("delegate_cart")
                        ctx.cart_result = result
                        tracer.record(name, {"product_id": product_id, "quantity": quantity}, result)
                    else:
                        result = {"ok": False, "error": f"Unknown tool: {name}"}
                        tracer.record(name, args, result)

            response = f"""Best option: {ctx.best['name']}
            Price: ${ctx.best['price']}
            Rating: {ctx.best['rating']}

            Pros: {ctx.pros.get(ctx.best['id'], '')}
            Cons: {ctx.cons.get(ctx.best['id'], '')}
            """
            # Если добавили товар
            if ctx.cart_result:
                if ctx.cart_result.get("ok"):
                    response += f"\nAdded to cart. Cart size: {ctx.cart_result.get('cart_size')}."
                else:
                    response += f"\nCould not add to cart: {ctx.cart_result.get('error', 'unknown error')}."
        else:
            response = "No suitable product found"

        return AgentResult(response, trace, ctx)

In [78]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")


OK 3.A
OK 3.B
OK 3.C
OK 3.D: context passed correctly, max_price is respected


In [83]:
_res3a.context.cons

{'p6': 'Higher price compared to other candidates, and lacks budget-friendly or portable tags that the user might prefer.',
 'p7': 'Lower rating than other candidates, and does not include premium or productivity features that may be desired for a wireless mouse.'}

In [81]:
_res3a.trace

['delegate_retriever',
 'delegate_pros',
 'delegate_cons',
 'delegate_ranker',
 'delegate_cart']

In [85]:
_res3a.response

'Best option: Logitech MX Master 3S\n            Price: $109\n            Rating: 4.8\n\n            Pros: Logitech MX Master 3S offers a stellar 4.8 rating and premium productivity features, all for just $109—well under the $120 budget. Its wireless design and premium build make it ideal for serious users.\n            Cons: Higher price compared to other candidates, and lacks budget-friendly or portable tags that the user might prefer.\n            \nAdded to cart. Cart size: 1.'